In [2]:
import numpy as np
from sklearn.metrics import roc_auc_score


def roc_auc(y_true, y_prob):
    """
    Compute Area Under the ROC Curve

    Parameters
    ----------
    y_true : array-like of shape (n_samples,)
        True binary labels (0 or 1).
    y_prob : array-like of shape (n_samples,)
        Predicted probabilities for the positive class.

    Returns
    -------
    float
        AUC-ROC score between 0 and 1.
    """
    return roc_auc_score(y_true, y_prob)

In [3]:
def roc_auc_with_ci(y_true, y_prob, n_bootstrap=1000):
    """
    Computes AUC-ROC with the boostrap confidence interval

    Parameters
    ----------
    y_true : array-like of shape (n_samples,)
        True binary labels (0 or 1).
    y_prob : array-like of shape (n_samples,)
        Predicted probabilities for the positive class.
    n_bootstrap : int, default 1000
        Number of bootstrap resamples.

    Returns
    -------
    auc : float                                                                         
        AUC-ROC score on the full dataset.                                              
    lower : float                                                                       
        Lower bound of the 95% confidence interval.
    upper : float                                                                       
        Upper bound of the 95% confidence interval.
                                                 
    """
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    n = len(y_true)
    scores = []

    for _ in range(n_bootstrap):
        # randomly sample n indicies WITH replacement
        indices = np.random.choice(n, size = n, replace = True)
        # calculate AUC on this sample and append to scores
        scores.append(roc_auc(y_true[indices], y_prob[indices]))
    auc = roc_auc(y_true, y_prob)
    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)
    return auc, lower, upper

In [4]:
import numpy as np
from sentinel.core.metrics.classification import roc_auc, roc_auc_with_ci

# fake data: 100 samples
np.random.seed(42)
y_true = np.random.randint(0, 2, 100)
y_prob = np.clip(y_true + np.random.normal(0, 0.3, 100), 0, 1)

# test basic AUC
print(roc_auc(y_true, y_prob))

# test with confidence interval
auc, lower, upper = roc_auc_with_ci(y_true, y_prob)
print(f"AUC: {auc:.3f} (95% CI: {lower:.3f}–{upper:.3f})")

0.9926948051948052
AUC: 0.993 (95% CI: 0.980–1.000)


In [5]:
from sentinel.core.metrics.classification import gini
print(gini(y_true, y_prob))

0.9853896103896105


In [6]:
  import numpy as np
  from sentinel.core.metrics.classification import roc_auc, roc_auc_with_ci, gini, ks_statistic

  np.random.seed(42)
  y_true = np.random.randint(0, 2, 100)
  y_prob = np.clip(y_true + np.random.normal(0, 0.3, 100), 0, 1)

  print(roc_auc(y_true, y_prob))
  auc, lower, upper = roc_auc_with_ci(y_true, y_prob)
  print(f"AUC: {auc:.3f} (95% CI: {lower:.3f}–{upper:.3f})")
  print(f"GINI: {gini(y_true, y_prob):.3f}")
  ks, pval = ks_statistic(y_true, y_prob)
  print(f"KS: {ks:.3f}  p-value: {pval:.4f}")

0.9926948051948052
AUC: 0.993 (95% CI: 0.980–1.000)
GINI: 0.985
KS: 0.942  p-value: 0.0000


In [7]:
from sentinel.core.metrics.classification import lift_table
print(lift_table(y_true, y_prob))

   decile   n  n_pos  pos_rate  cumulative_pos_pct    lift
0       1  10     10       1.0              0.1786  1.7857
1       2  10     10       1.0              0.3571  1.7857
2       3  10     10       1.0              0.5357  1.7857
3       4  10     10       1.0              0.7143  1.7857
4       5  10      9       0.9              0.8750  1.6071
5       6  10      6       0.6              0.9821  1.0714
6       7  10      1       0.1              1.0000  0.1786
7       8  10      0       0.0              1.0000  0.0000
8       9  10      0       0.0              1.0000  0.0000
9      10  10      0       0.0              1.0000  0.0000


In [8]:
from sentinel.core.metrics.calibration import brier_score, expected_calibration_error, reliability_diagram_data

print(f"Brier Score: {brier_score(y_true, y_prob):.4f}")
print(f"ECE: {expected_calibration_error(y_true, y_prob):.4f}")
fop, mpv = reliability_diagram_data(y_true, y_prob)
print(f"Reliability bins - predicted: {mpv.round(2)}")
print(f"Reliability bins - actual:    {fop.round(2)}")

Brier Score: 0.0442
ECE: 0.1625
Reliability bins - predicted: [0.01 0.14 0.25 0.33 0.44 0.56 0.66 0.74 0.84 0.99]
Reliability bins - actual:    [0.   0.   0.17 0.   0.5  1.   0.8  1.   1.   1.  ]


In [9]:
from sentinel.core.reporting.html_report import generate_report
                                                                                      
generate_report(
    metrics={'auc': 0.82, 'mae': 3.4, 'psi': 0.07},                                 
    model_name='credit_risk_v2',                                                    
    validation_date='2026-03-19',
    dataset_info={'n_observations': 5000, 'date_range': '2024-2025'},               
    output_path='test_report.html'                                                  
)                                                                                   
print('done')  

done


In [10]:
generate_report(
    metrics={'auc': 0.82, 'mae': 3.4},
    model_name='credit_risk_v2',
    validation_date='2026-03-19',
    dataset_info={'n_observations': 5000, 'date_range': '2024-2025'},
    output_path='test_report.html',
    drift={'psi': 0.07, 'flag': 'stable'},
    fairness={
        'disparate_impact_ratio': 0.85,
        'demographic_parity_difference': -0.04,
        'equalized_odds_gap': 0.03
    },
    challenger={
        'champ_auc': 0.82,
        'chal_auc': 0.86,
        'challenger_wins': True
    }
)
print('done')

done


In [15]:
from sentinel.core.reporting.sr11_7 import sr11_7_report

sr11_7_report(
    model_name="credit_risk_v2",
    validation_date="2026-03-27",
    dataset_info={"n_observations": 5000, "date_range": "2024-2025"},
    performance_metrics={
        "AUC-ROC": 0.82,
        "KS Statistic": 0.44,
        "Gini": 0.64,
        "Brier Score": 0.12,
        "PSI": 0.07
    },
    challenger_results={
        "champ_auc": 0.82,
        "chal_auc": 0.86,
        "z_statistic": 2.14,
        "p_value": 0.032,
        "challenger_wins": True
    },
    narrative={
        "model_purpose": "Logistic regression model predicting 12-month default probability for consumer credit applications.",
        "conceptual_soundness": "Model methodology is consistent with industry standard credit scoring practice.",
        "limitations": "Model was trained on 2020-2023 data and may not reflect post-2024 economic conditions."
    },
    output_path="sr11_7_test.html"
)
print("done")

done


In [16]:
from sentinel.core.registry.model_registry import init_db, register_model, get_model

init_db()

register_model(
    model_id="credit_risk_v2",
    model_name="Credit Risk Model V2",
    owner="Seth Gannon",
    model_type="classification",
    purpose="12-month default probability",
    tier=1,
    validation_status="pending"
)

record = get_model("credit_risk_v2")
print(record.model_name, record.tier, record.validation_status)

Credit Risk Model V2 1 pending


In [17]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sentinel import Validator                                                      
                                            
X, y = make_classification(n_samples=1000, random_state=42)                         
model = LogisticRegression().fit(X, y)
                                                                                    
val = Validator(model, X, y)            
report = val.run()                                                                  
report.save("validator_test.html", model_name="Test Model",                         
validation_date="2026-03-27")               
print("done") 

ImportError: cannot import name 'Validator' from 'sentinel' (/Users/sethgannon/Projects/sentinel/sentinel/__init__.py)